In [ ]:
import pickle
import numpy as np
from auditing_utils import get_privacy_profile
import matplotlib.pyplot as plt
from tueplots import bundles
rc = bundles.iclr2024(usetex=False)

In [ ]:
## Compute noise multiplier for target epsilon under substitute relation
from gdpnum.subst_prv import PoissonSubsampledGaussianMechanismSubstitute
from prv_accountant import PRVAccountant, PoissonSubsampledGaussianMechanism

target_delta = 1e-5
q = 1.0
T = 500
sigma = 22.37
K = 10
accounting_epsilons_subst = np.zeros(T//K + 1)
subst_prv = PoissonSubsampledGaussianMechanismSubstitute(sampling_probability=q, noise_multiplier=sigma)
for t in range(1,T+1,K):
    accountant = PRVAccountant(prvs=subst_prv, max_self_compositions=t, eps_error=1e-3, delta_error=1e-10)
    _, _, eps_up_s = accountant.compute_epsilon(delta=target_delta, num_self_compositions=t)
    accounting_epsilons_subst[t//K] = eps_up_s
accountant = PRVAccountant(prvs=subst_prv, max_self_compositions=T, eps_error=1e-3, delta_error=1e-10)
_, _, eps_up_s = accountant.compute_epsilon(delta=target_delta, num_self_compositions=T)
accounting_epsilons_subst[-1] = eps_up_s


accounting_epsilons_ar = np.zeros(T//K + 1)
ar_prv = PoissonSubsampledGaussianMechanism(sampling_probability=q, noise_multiplier=sigma)
for t in range(1,T+1,K):
    accountant = PRVAccountant(prvs=ar_prv, max_self_compositions=t, eps_error=1e-3, delta_error=1e-10)
    _, _, eps_up_s = accountant.compute_epsilon(delta=target_delta, num_self_compositions=t)
    accounting_epsilons_ar[t//K] = eps_up_s
accountant = PRVAccountant(prvs=ar_prv, max_self_compositions=T, eps_error=1e-3, delta_error=1e-10)
_, _, eps_up_s = accountant.compute_epsilon(delta=target_delta, num_self_compositions=T)
accounting_epsilons_ar[-1] = eps_up_s


In [ ]:
with open(f"auditing_results_q_{q}_sigma_{sigma}.pkl","rb") as f:
    attack_data = pickle.load(f)

attack_scores, attack_gts = attack_data["attack_scores"], attack_data["gts"]
T = attack_scores.shape[1]
attack_data["gdp_cp_lb"] = np.zeros(T//K + 1)
for t in range(0,T,K):
    _,_,_,epsilon_gdp_lbs = get_privacy_profile(delta =1e-5,
                                                outputs=attack_scores[:,t],
                                                gt=attack_gts)

    attack_data["gdp_cp_lb"][t//K] = max(epsilon_gdp_lbs)

_,_,_,epsilon_gdp_lbs_final = get_privacy_profile(delta =1e-5,
                                                outputs=attack_scores[:,-1],
                                                gt=attack_gts)
attack_data["gdp_cp_lb"][-1] =  max(epsilon_gdp_lbs_final)     

In [ ]:
with plt.rc_context(rc):
    fig,ax = plt.subplots(1,1)
    ax.plot(accounting_epsilons_subst, attack_data["gdp_cp_lb"], label=r"Crafted Gradient Canaries")
    ax.plot(accounting_epsilons_subst, accounting_epsilons_subst, "k--", label=r"$\varepsilon_S$ (Accounting)")
    ax.plot(accounting_epsilons_subst, accounting_epsilons_ar, "r--", label=r"$\varepsilon_{AR}$ (Accounting)")
    ax.set(xlabel=r"$\varepsilon_S$ (Accounting)", ylabel=r"$\varepsilon$")
    ax.legend()
    ax.set_box_aspect(1)
    plt.show()